In [94]:
import pandas as pd
import glob
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import random

In [95]:
folder_r = r"C:\Arjun\Thesis\data\20200421_170039-sunset1\filtered chunks\subsampled"
folder_q= r"C:\Arjun\Thesis\data\20200422_172431-sunset2\subsampling"
NUM_MATCHES = 50
MATCH_DIST = 10 
ref_length_max = 15  # Maximum length of reference sequence in seconds
query_length_max = 3
exp_no = "2"

In [96]:

def unix_to_brisbane(unix_time):
    """Convert UNIX timestamp to Brisbane time (UTC+10)"""
    # Brisbane is UTC+10 (no daylight saving)
    brisbane_time = datetime.fromtimestamp(unix_time, tz=timezone.utc) + pd.Timedelta(hours=10)
    return brisbane_time.strftime('%Y-%m-%d %H:%M:%S')

In [97]:
def get_timestamps(folder_path):
    """Extract 4th column timestamps from all CSVs in folder"""
    timestamps = []
    for csv_file in glob.glob(f"{folder_path}/*.csv"):
        df = pd.read_csv(csv_file)
        ts_col = df.iloc[:, 3]  # 4th column (0-indexed)
        timestamps.extend(ts_col.dropna().tolist())
    return set(timestamps)

In [98]:
ts_r = get_timestamps(folder_r)
ts_q = get_timestamps(folder_q)



In [99]:
print(f"ref {unix_to_brisbane(1587452832)}")
print(f"query {unix_to_brisbane(1587540502)}")



ref 2020-04-21 17:07:12
query 2020-04-22 17:28:22


In [100]:
min_r = unix_to_brisbane(min(ts_r))
max_r = unix_to_brisbane(max(ts_r))
min_q = unix_to_brisbane(min(ts_q))
max_q = unix_to_brisbane(max(ts_q))
print(f"Ref - Min: {min_r}, Max: {max_r}")
print(f" Ref {int((max(ts_r) - min(ts_r))/60)} min {int((max(ts_r) - min(ts_r)) % 60)} sec  ") 
print(f"query - Min: {min_q}, Max: {max_q}s")
print(f" query {int((max(ts_q) - min(ts_q))/60)} min {int((max(ts_q) - min(ts_q)) % 60)} sec  ") 

Ref - Min: 2020-04-21 17:03:03, Max: 2020-04-21 17:15:07
 Ref 12 min 3 sec  
query - Min: 2020-04-22 17:24:21, Max: 2020-04-22 17:35:12s
 query 10 min 51 sec  


In [101]:
ref_gps = pd.read_csv(r"C:\Arjun\Thesis\data\20200421_170039-sunset1\sunset1_gps_v2.csv")
query_gps = pd.read_csv(r"C:\Arjun\Thesis\data\20200422_172431-sunset2\sunset2_gps_v2.csv")

In [102]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Earth radius in meters
    R = 6371000

    # Convert latitude and longitude from degrees to radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    # Calculate differences between latitudes and longitudes
    d_lat = lat2_rad - lat1_rad
    d_lon = lon2_rad - lon1_rad

    # Haversine formula
    a = np.sin(d_lat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(d_lon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    # Calculate the distance
    distance = R * c

    return distance

In [103]:
ref_lat = ref_gps['latitude']
ref_lon = ref_gps['longitude']
ref_ts = ref_gps['elapsed_time_ts']  # timestamp column

query_lat = query_gps['latitude']
query_lon = query_gps['longitude']
query_ts = query_gps['elapsed_time_ts']


# Use GPS elapsed_time_ts instead of CSV timestamps
min_r = unix_to_brisbane(min(ref_ts))
max_r = unix_to_brisbane(max(ref_ts))
min_q = unix_to_brisbane(min(query_ts))
max_q = unix_to_brisbane(max(query_ts))
print(f"Ref - Min: {min_r}, Max: {max_r}, period: {(max(ref_ts) - min(ref_ts))/60} min")
print(f"query - Min: {min_q}, Max: {max_q}, period: {(max(query_ts) - min(query_ts))/60} min")

Ref - Min: 2020-04-21 17:03:02, Max: 2020-04-21 17:14:56, period: 11.9 min
query - Min: 2020-04-22 17:24:32, Max: 2020-04-22 17:35:02, period: 10.5 min


In [104]:
'''window_r = set(ts_r)  # timestamps from first folder's CSVs
window_q = set(ts_q)'''
window_r = set(ref_ts)  # timestamps from first folder's CSVs
window_q = set(query_ts)

In [105]:
def find_nearest_gps_match(ref_lat, ref_lon, ref_ts, query_lat, query_lon, query_ts, window_r, window_q, match_dist):
    """Find the nearest GPS match in query for a single reference point (only if within match_dist meters)"""
    best_match = None   
    for j, (qlat, qlon, qts) in enumerate(zip(query_lat, query_lon, query_ts)):
        dist = haversine_distance(ref_lat, ref_lon, qlat, qlon)
        
        if dist <= match_dist:
            best_match = {
                'ref_lat': ref_lat, 
                'ref_lon': ref_lon,
                'query_lat': qlat, 
                'query_lon': qlon,
                'ref_ts': ref_ts,
                'ref_time': unix_to_brisbane(ref_ts),
                'query_ts': qts,
                'query_time': unix_to_brisbane(qts),
                'distance_m': dist,
            
            }
            return best_match
    
    # Only return match if distance is within threshold

    return None

In [106]:
# Randomly select NUM_MATCHES from reference points
print(f"\nRandomly selecting {NUM_MATCHES} reference points from {len(ref_lat)} total points...")
random.seed(42)  # Set seed for reproducibility (optional)
random_indices = random.sample(range(len(ref_lat)), min(NUM_MATCHES, len(ref_lat)))

matches = []
print(f"Finding nearest GPS matches for each selected reference point (threshold: {MATCH_DIST}m)...")
print("-" * 60)

for idx in random_indices:
    match = find_nearest_gps_match(
        ref_lat.iloc[idx], ref_lon.iloc[idx], ref_ts.iloc[idx],
        query_lat, query_lon, query_ts,
        window_r, window_q, MATCH_DIST
    )
    if match:
        matches.append(match)
        print(f"Reference point {idx}: Distance = {match['distance_m']:.2f}")
    else:
        print(f"Reference point {idx}: No match found within {MATCH_DIST}m")

print("-" * 60)
print(f"COMPLETED! Found {len(matches)} matches within {MATCH_DIST}m threshold")


Randomly selecting 50 reference points from 578 total points...
Finding nearest GPS matches for each selected reference point (threshold: 10m)...
------------------------------------------------------------
Reference point 114: Distance = 7.84
Reference point 25: Distance = 0.83
Reference point 281: Distance = 1.07
Reference point 250: Distance = 6.11
Reference point 228: Distance = 2.11
Reference point 142: Distance = 2.26
Reference point 104: Distance = 5.81
Reference point 558: Distance = 3.39
Reference point 89: Distance = 0.55
Reference point 432: Distance = 2.90
Reference point 32: Distance = 2.64
Reference point 30: Distance = 1.73
Reference point 95: Distance = 4.71
Reference point 223: Distance = 4.75
Reference point 238: Distance = 7.34
Reference point 517: Distance = 2.74
Reference point 27: Distance = 0.63
Reference point 574: Distance = 6.26
Reference point 203: Distance = 9.73
Reference point 429: Distance = 2.49
Reference point 225: Distance = 4.33
Reference point 459: 

In [107]:
# Results
if len(matches) > 0:
    matches_df = pd.DataFrame(matches)
   
    print(f"\n--- MATCH SUMMARY ---")
    print(f"Total matches found: {len(matches)}")
    
    print(f"Average match distance: {matches_df['distance_m'].mean():.2f} meters")
    print(f"Min distance: {matches_df['distance_m'].min():.2f} meters")
    print(f"Max distance: {matches_df['distance_m'].max():.2f} meters")
    
    
    # Save all matches to CSV
    matches_df.to_csv(f"exp/gps_matches_{exp_no}.csv", index=False)
    print(f"\n✓ Saved all {len(matches)} matches to 'gps_matches_{NUM_MATCHES}_samples.csv'")
    
    # Also save only valid matches (timestamps within both windows)
    
    #matches_df.to_csv(f"valid_matches.csv", index=False)
    
else:
    print("No matches found within threshold!")
    print(f"Try increasing MATCH_DIST (currently {MATCH_DIST}m) or check if GPS trajectories overlap")


--- MATCH SUMMARY ---
Total matches found: 50
Average match distance: 4.29 meters
Min distance: 0.55 meters
Max distance: 9.73 meters

✓ Saved all 50 matches to 'gps_matches_50_samples.csv'


In [108]:
# Load your GPS file (adjust path as needed)
'''gps_file = pd.read_csv(r"C:\Arjun\Thesis\data\20200422_172431-sunset2\sunset2_gps_v2.csv")

# Extract coordinates
latitudes = gps_file['latitude']
longitudes = gps_file['longitude']

# Get first coordinate
first_lat = latitudes.iloc[0]
first_lon = longitudes.iloc[0]

# Calculate distances from first point to all others
distances = []
for i in range(len(latitudes)):
    dist = haversine_distance(first_lat, first_lon, latitudes.iloc[i], longitudes.iloc[i])
    distances.append(dist)

# Add distances to dataframe
gps_file['distance_from_first_m'] = distances

# Display results
print(f"First coordinate: ({first_lat}, {first_lon})")
print(f"Total points: {len(distances)}")
print("\nFirst 10 distances from first point:")
for i in range(len(distances)):
    print(f"Point {i}: {distances[i]:.2f} meters")

# Show summary
print(f"\n--- Summary ---")
print(f"Max distance from first point: {max(distances):.2f} meters")
print(f"Min distance from first point: {min(distances):.2f} meters")
print(f"Average distance: {sum(distances)/len(distances):.2f} meters")

# Optional: Save to new CSV
gps_file.to_csv("gps_with_distances_from_first.csv", index=False)
print("\n✓ Saved to 'gps_with_distances_from_first.csv'")'''

<>:2: SyntaxWarning: invalid escape sequence '\A'
<>:2: SyntaxWarning: invalid escape sequence '\A'
C:\Users\arj_s\AppData\Local\Temp\ipykernel_21096\781749876.py:2: SyntaxWarning: invalid escape sequence '\A'
  '''gps_file = pd.read_csv(r"C:\Arjun\Thesis\data\20200422_172431-sunset2\sunset2_gps_v2.csv")


'gps_file = pd.read_csv(r"C:\\Arjun\\Thesis\\data\x8200422_172431-sunset2\\sunset2_gps_v2.csv")\n\n# Extract coordinates\nlatitudes = gps_file[\'latitude\']\nlongitudes = gps_file[\'longitude\']\n\n# Get first coordinate\nfirst_lat = latitudes.iloc[0]\nfirst_lon = longitudes.iloc[0]\n\n# Calculate distances from first point to all others\ndistances = []\nfor i in range(len(latitudes)):\n    dist = haversine_distance(first_lat, first_lon, latitudes.iloc[i], longitudes.iloc[i])\n    distances.append(dist)\n\n# Add distances to dataframe\ngps_file[\'distance_from_first_m\'] = distances\n\n# Display results\nprint(f"First coordinate: ({first_lat}, {first_lon})")\nprint(f"Total points: {len(distances)}")\nprint("\nFirst 10 distances from first point:")\nfor i in range(len(distances)):\n    print(f"Point {i}: {distances[i]:.2f} meters")\n\n# Show summary\nprint(f"\n--- Summary ---")\nprint(f"Max distance from first point: {max(distances):.2f} meters")\nprint(f"Min distance from first point: 

In [109]:
import json
config = {
    "experiment_name": f"dtw_test_{NUM_MATCHES}_{exp_no}",
    "data": {
        "query_folder": folder_q,  # Using your existing folder variables
        "ref_folder": folder_r
    },
    "max_ref_l": ref_length_max,
    "max_query_l":query_length_max,
    "pairs": []
}

In [110]:
for idx, row in matches_df.iterrows():
    # Get timestamps
    ref_ts = row['ref_ts']
    query_ts = row['query_ts']
    
    # Randomize window lengths (less than max)
    import random
    ref_len = round(random.uniform(5, ref_length_max), 1)
    query_len = round(random.uniform(0.5, query_length_max), 1)

    # Calculate window starts (ensure timestamp is not at midpoint)
    # Random offset between 20% and 80% of window length
    ref_offset_pct = random.uniform(0.2, 0.8)
    query_offset_pct = random.uniform(0.2, 0.8)
    
    ref_start = round(ref_ts - (ref_len * ref_offset_pct), 1)
    ref_end = round(ref_ts + (ref_len * (1 - ref_offset_pct)), 1)
    
    query_start = round(query_ts - (query_len * query_offset_pct), 1)
    query_end = round(query_ts + (query_len * (1 - query_offset_pct)), 1)

    # Create pair config
    pair = {
        "pair_id": idx + 1,
        "query_start": query_start,
        "query_end": query_end,
        "ref_start": ref_start,
        "ref_end": ref_end,
        "ref_l": ref_len,
        "query_l": query_len,
        "ref_match": ref_ts,
        "q_match": query_ts,
        "gps_q": [round(row['query_lat'], 6), round(row['query_lon'], 6)],  # GPS to 6 decimals
        "type": "good"
    }
    config["pairs"].append(pair)

with open(f'exp/{exp_no}_mdtw_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Generated config with {len(config['pairs'])} pairs")
print(f"Saved to exp/{exp_no}_mdtw_config.json")

✓ Generated config with 50 pairs
Saved to exp/2_mdtw_config.json
